# Replicando o Sean

> Sean Carlisto De Alvarenga, Sylvio Barbon, Rodrigo Sanches Miani, Michel Cukier, e Bruno Bogaz Zarpelão. "***Process Mining and Hierarchical Clustering to Help Intrusion Alert Visualization***". Computers & Security 73 (março de 2018): 474–91. https://doi.org/10.1016/j.cose.2017.11.021.

### O Sean escolhe dias pelo ano todo

### Aqui temos 10 dias (Lab, 16 a 25/nov) + 23 dias (Real, 27/jun a 20/jul) disponíveis

### Sendo assim, agregaremos juntando *ambos* os cenários

### "Alertas" COMISET = linhas com rótulo MITRE (~1,7M no Lab e ~630k no Real)

### Grupos com só 1 alerta ou todos iguais são descartados = queremos ataques multi-fase

### Ressalva técnica (na agregação): Ward pressupõe euclidiana, Jaccard não é. Sean usou assim mesmo, aqui faremos igual.

# Imports e Configs

In [86]:
%xmode plain
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

Exception reporting mode: Plain


In [87]:
import os
import time
from contextlib import contextmanager
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [88]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [89]:
from util import cron

## Conexão DuckDB

In [90]:
conn = duckdb.connect()
conn.execute("SET threads = 15")
conn.execute("SET memory_limit = '30GB'")
conn.execute("SET checkpoint_threshold = '1GB'")

In [91]:
RAW = {
    "lab": Path("Parquet/lab_events.parquet"),
    "real": Path("Parquet/real_events.parquet"),
}
OUT = Path("Sean")  # artefatos deste notebook
OUT.mkdir(exist_ok=True)

# 1. Alertas

### No Sean, a entrada são alertas de IDS 

### Aqui são eventos com regra Sysmon (`rule_technique_id` presente)

### Pegamos só esses, de todos os dias, só as colunas que interessam

### Tal agregação já seria um "achatamento à la Sean"

### Importante: Direção dos IPs

### Sean usa tráfego de entrada, então o `src_ip` é o atacante

### Aqui, no Sysmon, o `src_ip` é o alvo. Temos que verificar a direção

### O campo `Initiated` tem que ser `false` = conexão veio de fora

In [92]:
# chave inexistente no MAP (do duckdb) vira NULL (cf. cobertura logo abaixo)
# "nome_nosso" : "nome_original"
CAMPOS = {
    "host": "host_name",
    "technique_id": "rule_technique_id",
    "technique_name": "rule_technique_name",
    "rule": "RuleName",
    "usr": "user_name",
    "proc_guid": "process_guid",
    "proc_name": "process_name",
    "proc_path": "process_path",
    "proc_parent_guid": "process_parent_guid",
    "proc_parent_name": "process_parent_name",
    "tgt_proc_guid": "target_process_guid",
    "tgt_proc_name": "target_process_name",
    "src_ip": "src_ip_addr",
    "dst_ip": "dst_ip_addr",
    "dst_port": "DestinationPort",
    "initiated": "Initiated",   # EID 3: true = saída, false = entrada
}

In [93]:
def sql_alertas(cen):
    # json_extract_string tira as aspas e decodifica os escapes (\\ -> \)
    cols = ",\n        ".join(
        f"""json_extract_string(_source['{k}'], '$') AS "{a}" """
        for a, k in CAMPOS.items()
    )
    return f"""
    SELECT
        _id,
        split_part(_index, '-', 4) AS canal,
        try_cast(json_extract_string(_source['@timestamp'], '$') AS TIMESTAMP) AS ts,
        try_cast(json_extract_string(_source['event_id'], '$') AS INTEGER)   AS event_id,
        {cols}
    FROM read_parquet('{RAW[cen]}')
    WHERE _source['rule_technique_id'] IS NOT NULL
    """

In [94]:
# lento: varredura full do arquivo bruto, pros 2 cenários
# feita só uma vez, dps confere se já existe o arquivo
for cen in ["lab", "real"]:
    dst = OUT / f"alertas_{cen}.parquet"
    if dst.exists():
        print(f"'{dst}' já existe, pulando")
        continue
    with cron(f"extração {cen}"):
        conn.execute(
            f"COPY ({sql_alertas(cen)}) TO '{dst}' (FORMAT PARQUET, CODEC 'ZSTD')"
        )

'Sean\alertas_lab.parquet' já existe, pulando
'Sean\alertas_real.parquet' já existe, pulando


# 1.1 Conferência
> Esperado: 1.713.709 (lab) e 631.606 (real)

> Números dos cenários:

LAB
* `maliciosos_lab   =  1_713_709`
* `total_linhas_lab =  49_914_325`
* `proporção_lab    =  3,43%`

REAL
* `maliciosos_real   = 631_606`
* `total_linhas_real = 202_304_794`
* `proporção_real    = 0,31%`

> Somando pra obter o N total desse notebook

* `maliciosos = 1_713_709   + 631_606     = 2.345.315`
* `total      = 49_914_325  + 202_304_794 = 252.219.119`

> `alertas` é um dict com dois elementos, um de cada cenário

> As chaves são o nome do cenário e os valores são dataframes (os df's dos alertas)

In [95]:
alertas = {
    c: pd.read_parquet(OUT / f"alertas_{c}.parquet", dtype_backend="pyarrow")
    for c in ["lab", "real"]
}
for c, df in alertas.items():
    print(
        f"{c:5s} {len(df):>10,} alertas | ts inválidos: {df['ts'].isna().sum()} | "
        f"hosts: {df['host'].nunique():>2} | {df['ts'].min().strftime('%d/%m/%Y')} a {df['ts'].max().strftime('%d/%m/%Y')}"
    )

lab    1,713,709 alertas | ts inválidos: 0 | hosts:  1 | 16/11/2022 a 25/11/2022
real     631,606 alertas | ts inválidos: 0 | hosts: 15 | 27/06/2022 a 20/07/2022


In [97]:
alertas["lab"].head(2)
alertas["lab"].shape

,_id,canal,ts,event_id,host,technique_id,technique_name,rule,usr,proc_guid,proc_name,proc_path,proc_parent_guid,proc_parent_name,tgt_proc_guid,tgt_proc_name,src_ip,dst_ip,dst_port,initiated
0,f70944eb67850c34d55c4f120e0abbcc99d3410c,sysmon,2022-11-16 19:59:51.007000,12,desktop-4pvps6e.phoenix.local,T1553.004,Install Root Certificate,"technique_id=T1553.004,technique_name=Install ...",louis,4FD6B357-412E-6375-CA00-000000002900,onedrive.exe,c:\users\louis\appdata\local\microsoft\onedriv...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,4a72999a2c943dea5e7694192dcb22a289ccffb4,sysmon,2022-11-16 19:59:50.377000,7,desktop-4pvps6e.phoenix.local,T1574.002,DLL Side-Loading,"technique_id=T1574.002,technique_name=DLL Side...",louis,4FD6B357-412E-6375-CA00-000000002900,onedrive.exe,c:\users\louis\appdata\local\microsoft\onedriv...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


(1713709, 20)

In [98]:
alertas["real"].head(2)
alertas["real"].shape

,_id,canal,ts,event_id,host,technique_id,technique_name,rule,usr,proc_guid,proc_name,proc_path,proc_parent_guid,proc_parent_name,tgt_proc_guid,tgt_proc_name,src_ip,dst_ip,dst_port,initiated
0,5b500a180e4bbf4fb6368bb65e3b5878ca17342b,sysmon,2022-06-27 06:54:22.310000,7,computer15.domain.es,T1055,Process Injection,"technique_id=T1055,technique_name=Process Inje...",system,89c3ec89-541e-62b9-600d-000000008200,tc.exe,c:\program files\common files\siemens\automati...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,0444eb0b3fa45a2f22db2f4eb47ec69d96954f22,sysmon,2022-06-27 06:54:25.590000,7,computer15.domain.es,T1055,Process Injection,"technique_id=T1055,technique_name=Process Inje...",system,89c3ec89-5421-62b9-610d-000000008200,tc.exe,c:\program files\common files\siemens\automati...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


(631606, 20)

# 1.2 Cobertura dos campos
> Por qual campo daria pra agrupar?

In [99]:
# `alertas` é um dict com dois elementos, um de cada cenário
# as chaves são o nome do cenário e os valores são dataframes (os df's dos alertas)
cob = pd.DataFrame({c: df.notna().mean() * 100 for c, df in alertas.items()}).T
cob
cob.shape

,_id,canal,ts,event_id,host,technique_id,technique_name,rule,usr,proc_guid,proc_name,proc_path,proc_parent_guid,proc_parent_name,tgt_proc_guid,tgt_proc_name,src_ip,dst_ip,dst_port,initiated
lab,100.00,100.00,100.00,100.00,100.00,100.00,99.99,100.00,95.53,100.00,100.00,100.00,0.26,0.26,4.47,4.47,68.62,68.62,68.62,68.62
real,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,91.44,100.00,100.00,100.00,0.55,0.55,8.56,8.56,0.27,0.27,0.27,0.27


(2, 20)

> Qual a presença dos campos por EID? E qual a cardinalidade de cada EID?

In [132]:
# cada EID (tipo de evento Sysmon) tem campos diferentes
ORIG_DEST = ["proc_guid", "tgt_proc_guid", "proc_parent_guid", "usr", "src_ip", "dst_ip", "initiated"]

def cob_eid(df):
    t = df.groupby("event_id")[ORIG_DEST].agg(lambda s: s.notna().mean() * 100)
    t.insert(0, "n", df.groupby("event_id").size())
    t = t.sort_values(by="n")
    t["n"] = t["n"].map('{:,d}'.format)
    return t

In [133]:
cob_eid(alertas["lab"])

,n,proc_guid,tgt_proc_guid,proc_parent_guid,usr,src_ip,dst_ip,initiated
event_id,,,,,,,,
18,107,100.00,0.00,0.00,100.00,0.00,0.00,0.00
15,199,100.00,0.00,0.00,100.00,0.00,0.00,0.00
1,"4,379",100.00,0.00,100.00,100.00,0.00,0.00,0.00
8,"6,422",100.00,100.00,0.00,0.00,0.00,0.00,0.00
7,"27,988",100.00,0.00,0.00,100.00,0.00,0.00,0.00
10,"70,181",100.00,100.00,0.00,0.00,0.00,0.00,0.00
12,"121,751",100.00,0.00,0.00,100.00,0.00,0.00,0.00
13,"151,553",100.00,0.00,0.00,100.00,0.00,0.00,0.00
11,"155,132",100.00,0.00,0.00,100.00,0.00,0.00,0.00


In [134]:
cob_eid(alertas["real"])

,n,proc_guid,tgt_proc_guid,proc_parent_guid,usr,src_ip,dst_ip,initiated
event_id,,,,,,,,
18,1,100.00,0.00,0.00,100.00,0.00,0.00,0.00
11,4,100.00,0.00,0.00,100.00,0.00,0.00,0.00
2,44,100.00,0.00,0.00,100.00,0.00,0.00,0.00
15,65,100.00,0.00,0.00,100.00,0.00,0.00,0.00
8,317,100.00,100.00,0.00,0.00,0.00,0.00,0.00
3,"1,676",100.00,0.00,0.00,99.88,100.00,100.00,100.00
1,"3,457",100.00,0.00,100.00,100.00,0.00,0.00,0.00
13,"18,672",100.00,0.00,0.00,100.00,0.00,0.00,0.00
12,"29,793",100.00,0.00,0.00,100.00,0.00,0.00,0.00


# 1.3 IPs

> Quantos IPs distintos e quais suas direções?

In [49]:
for c, df in alertas.items():
    ip = df[df["src_ip"].notna()]
    pares = ip[["src_ip", "dst_ip"]].drop_duplicates()
    
    print(
        f"\n{"="*50}\n"
        f"{c.upper()}: {len(ip):,} alertas com IP ({len(ip) / len(df):.1%}) \n"
        f"src distintos: {ip['src_ip'].nunique()} \n"
        f"dst distintos: {ip['dst_ip'].nunique()} \n"
        f"pares src-dst: {len(pares)}"
    )

    print("\nDireção")
    display(ip["initiated"].value_counts(dropna=False))   # direção
    print("\nTop 10 Origem")
    display(ip["src_ip"].value_counts().head(10))         # quem mais aparece como origem
    print("\nTop 10 Processos Iniciadores")
    display(ip["proc_name"].value_counts().head(10))      # que processo faz as conexões


LAB: 1,175,997 alertas com IP (68.6%) 
src distintos: 57 
dst distintos: 154 
pares src-dst: 340

Direção


initiated
true     1175410
false        587
Name: count, dtype: int64[pyarrow]


Top 10 Origem


src_ip
192.168.137.132    420144
192.168.47.142     325887
192.168.182.131    121669
192.168.137.131     97631
192.168.47.147      76820
192.168.235.150     29939
192.168.137.148     18401
192.168.253.141     15619
192.168.50.131       8584
192.168.253.135      7644
Name: count, dtype: int64[pyarrow]


Top 10 Processos Iniciadores


...

initiated
true     1442
false     234
Name: count, dtype: int64[pyarrow]


Top 10 Origem


src_ip
192.168.194.132    556
127.0.0.1          206
172.20.2.109       198
172.20.2.116       135
172.20.2.115        95
172.20.2.105        82
172.20.2.110        61
172.20.2.117        60
172.20.2.112        59
172.20.2.106        45
Name: count, dtype: int64[pyarrow]


Top 10 Processos Iniciadores


proc_name
printerserviceserverconsole.exe    338
msmpeng.exe                        311
nslookup.exe                       217
powershell.exe                     175
su-repair.exe                      162
cmd.exe                            153
gc_print_data_service.exe          124
sophosupdate.exe                   111
certutil.exe                        20
sqlceip.exe                         11
Name: count, dtype: int64[pyarrow]

> **Quase não há tráfego de entrada: 587 (de 1,175,997) no LAB, e 234 (de 1,676) no REAL**

> **A replicação literal vai trabalhar com muito pouco dado.**

# 1.4 Direção do tráfego
> Sean usa só tráfego de entrada (de fora pra dentro, *inbound*).

> Quem são as origens de entrada e quais serviços (portas) são alvo?

In [140]:
for c, df in alertas.items():
    ip = df[df["src_ip"].notna()]
    print(f"\n{c.upper()}  (false = entrada, true = saída)")
    display(
        ip.groupby("initiated").agg(
            alertas=("src_ip", "size"),
            src=("src_ip", "nunique"),
            dst=("dst_ip", "nunique"),
            portas=("dst_port", "nunique"),
        )
    )
    entrada = ip[ip["initiated"] == "false"]
    entrada.value_counts(["src_ip", "dst_ip", "dst_port", "proc_name"])


LAB  (false = entrada, true = saída)


,alertas,src,dst,portas
initiated,,,,
false,587,20,13,16
true,1175410,39,143,52


src_ip           dst_ip           dst_port  proc_name                
10.10.234.120    10.10.234.129    3389      svchost.exe                  239
10.10.234.128    10.10.234.129    3389      svchost.exe                   90
127.0.0.1        127.0.0.1        6000      xwin_mobax.exe                36
10.10.234.1      10.10.234.129    3389      svchost.exe                   35
10.10.234.131    10.10.234.129    3389      svchost.exe                   35
192.168.182.128  192.168.182.131  3389      svchost.exe                   22
10.10.234.133    10.10.234.129    3389      svchost.exe                   20
192.168.16.128   192.168.16.130   3389      svchost.exe                   16
10.10.117.130    10.10.117.130    3389      svchost.exe                   14
192.168.5.139    192.168.5.137    3389      svchost.exe                   14
192.168.16.133   192.168.16.132   3389      svchost.exe                    9
192.168.253.129  192.168.253.141  3389      svchost.exe                    7
192.16


REAL  (false = entrada, true = saída)


,alertas,src,dst,portas
initiated,,,,
false,234,2,2,95
true,1442,17,84,225


src_ip           dst_ip         dst_port  proc_name                      
192.168.194.132  192.168.194.2  53        nslookup.exe                       80
                                          cmd.exe                            54
127.0.0.1        127.0.0.1      4536      printerserviceserverconsole.exe     7
                                4709      gc_print_data_service.exe           1
                                33523     gc_print_data_service.exe           1
                                                                             ..
                                1638      gc_print_data_service.exe           1
                                1628      gc_print_data_service.exe           1
                                1630      gc_print_data_service.exe           1
                                1632      gc_print_data_service.exe           1
                                1640      gc_print_data_service.exe           1
Name: count, Length: 96, dtype: int64

# 1.5 Por que tantos IPs de Origem no LAB?

> Os 57 IPs de origem do LAB: as sub-redes mudam por dia? (hipótese: VMs em redes diferentes por sessão)

In [143]:
with cron("crosstab pra detectar sub-redes diferentes ao longo dos dias/sessões"):
    lab = alertas["lab"]
    saida = lab[lab["initiated"] == "true"]
    rede = saida["src_ip"].str.extract(r"^(?P<rede>\d+\.\d+\.\d+)\.", expand=False)   # os 3 primeiros blocos do IP
    pd.crosstab(saida["ts"].dt.strftime("%d/%m"), rede)

rede,10.10.117,10.10.234,127.0.0,192.168.137,192.168.16,192.168.182,192.168.21,192.168.232,192.168.235,192.168.253,192.168.47,192.168.5,192.168.50,192.168.75,192.168.88
ts,,,,,,,,,,,,,,,
16/11,4,8,21,97630,0,0,0,3,0,324,77,0,0,2,86
17/11,6,0,9,18165,15,0,0,0,0,2236,325809,367,0,2,2874
18/11,51,165,9,773,10,109500,49,16,0,12273,3592,0,0,4000,1040
19/11,131,2,10,1778,2,3833,1223,0,24008,0,8152,0,0,0,12801
20/11,53,2,4,269430,0,129,7480,0,0,0,68668,0,0,2699,270
21/11,10,0,3,148936,1,4161,280,0,323,0,0,0,6,0,0
22/11,0,0,0,0,37,4046,0,0,1017,0,0,0,0,0,1
23/11,4,0,0,53,415,0,0,0,3890,0,0,0,0,0,0
24/11,52,0,0,180,871,0,0,0,0,14139,0,2402,5128,0,0



[tempo] crosstab pra detectar sub-redes diferentes ao longo dos dias/sessões: 7.8s



# 2. Higiene das técnicas
IDs sujos ("1210", "T", "Port Monitors") e de versões antigas do ATT&CK (por ex., real usa T1130, lab usa T1553.004 para a mesma coisa)

> Pra entender esse regex com capture group nomeado (o `P` é de Python)

In [113]:
alertas["lab"]["rule"][1]

'technique_id=T1574.002,technique_name=DLL Side-Loading'

In [116]:
# `expand=True` faria retornar um DataFrame com uma coluna com o nome do capture group
with cron("regex mais pesado"):
    alertas["lab"]["rule"].str.extract(r"technique_id=(?P<tid>[^,]+)", expand=False)
    alertas["real"]["rule"].str.extract(r"technique_id=(?P<tid>[^,]+)", expand=False)

0          T1553.004
1          T1574.002
2          T1574.002
3          T1055.001
4          T1055.001
             ...    
1713704    T1055.001
1713705        T1055
1713706        T1055
1713707        T1055
1713708        T1543
Name: tid, Length: 1713709, dtype: string[pyarrow]

0         T1055
1         T1055
2         T1055
3         T1055
4         T1055
          ...  
631601    T1055
631602    T1055
631603    T1055
631604    T1055
631605    T1055
Name: tid, Length: 631606, dtype: string[pyarrow]


[tempo] regex mais pesado: 0.5s



> Conferir ID contra o RuleName

In [117]:
for c, df in alertas.items():
    df["technique_id_rule"] = df["rule"].str.extract(r"technique_id=(?P<tid>[^,]+)", expand=False)
    dif = df["technique_id"].ne(df["technique_id_rule"]).fillna(True)
    print(f"{c.upper():>4}: {dif.sum():,} alertas com `technique_id` diferente do RuleName")

 LAB: 0 alertas com `technique_id` diferente do RuleName
REAL: 0 alertas com `technique_id` diferente do RuleName


> Quem dispara os IDs suspeitos

In [119]:
SUSPEITOS = ["T", "T1064", "T1175", "T1089"]
# T     -> sem número
# T1064 -> técnica deprecated
# T1175 -> ambíguo entre COM e DCOM
# T1089 -> ID e nome não batiam

for c, df in alertas.items():
    s = df[df["technique_id"].isin(SUSPEITOS)]
    print(f"Cenário: {c.upper():>4}")
    s.value_counts(["technique_id", "technique_name", "event_id", "proc_name"])
    print()

Cenário:  LAB


Series([], Name: count, dtype: int64)


Cenário: REAL


technique_id  technique_name                              event_id  proc_name                      
T1175         Component Object Model and Distributed COM  7         sdxhelper.exe                      39
T1089         Drive-by Compromise                         15        explorer.exe                       25
                                                                    chrome.exe                         19
T1175         Component Object Model and Distributed COM  7         msoia.exe                          11
T1064         Scripting                                   11        pwsh.exe                            3
              Windows Scripting Host Component            7         wscript.exe                         3
T1089         Drive-by Compromise                         15        firefox.exe                         2
T1064         Windows Scripting Host Component            7         regsvr32.exe                        2
T1175         Component Object Model and Distributed

> Dicionário de correção

In [148]:
# ID sujo ou antigo -> ID atual
FIX = {
    "1210": "T1210",
    "Port Monitors": "T1547.010",
    "T1546.0082": "T1546.008",
    "T1547.0102": "T1547.010",
    "T1055; Possible Cobalt Strike post-exploitation jobs.": "T1055",
    "T1086": "T1059.001",  # PowerShell
    "T1093": "T1055.012",  # Process Hollowing
    "T1130": "T1553.004",  # Install Root Certificate
    "T1073": "T1574.002",  # DLL Side-Loading
    "T1099": "T1070.006",  # Timestomp
    "T1170": "T1218.005",  # Mshta
    "T1117": "T1218.010",  # Regsvr32
    "T1063": "T1518.001",  # Security Software Discovery
    "T1031": "T1543.003",  # Modify Existing Service
    "T1064": "T1059.005",  # Scripting -> Visual Basic (wscript/cscript)                ; cf. acima
    "T1175": "T1559.001",  # COM (se for DCOM seria T1021.003)                          ; cf. acima
    "T1089": "T1189",       # ID diz Disabling Security Tools, nome diz Drive-by (T1189) ; cf. acima
}

> Aplicação da correção e diferenças (oq sobrou)

> A nova coluna `technique` é o nome normalizado/corrigido

In [149]:
PADRAO = r"^T\d{4}(\.\d{3})?$"

In [150]:
with cron("regex e diferenças pós-correção rótulos MITRE"):
    for c, df in alertas.items():
        # como em 2. Higiene tivemos "0 alertas com `technique_id` diferente do RuleName" essas duas não precisamos mais
        # t = df["technique_id"].replace(FIX)
        # ok = t.str.match(PADRAO).fillna(False)
        df["technique"] = df["technique_id"].replace(FIX)                                        # normalizada: é a atividade
        df["technique_parent"] = df["technique"].str.extract(r"^(?P<pai>T\d{4})", expand=False)  # sem subtécnica
        sobra = df.loc[~df["technique"].str.match(PADRAO).fillna(False), "technique"].value_counts(dropna=False)
        print(
            f"{c.upper():>4}: {df['technique_id'].nunique()} -> {df['technique'].nunique()} técnicas "
            f"| {df['technique_parent'].nunique()} pais | fora do padrão: {sobra.to_dict()}"
        )
    
    a, b = (set(alertas[c]["technique_id"].dropna()) for c in ["lab", "real"])
    A, B = (set(alertas[c]["technique"].dropna()) for c in ["lab", "real"])
    print(f"\ntécnicas em comum LAB e REAL: {len(a & b)} -> {len(A & B)}")

 LAB: 53 -> 50 técnicas | 35 pais | fora do padrão: {'T': 55}
REAL: 57 -> 53 técnicas | 33 pais | fora do padrão: {'T': 1}

técnicas em comum LAB e REAL: 37 -> 40

[tempo] regex e diferenças pós-correção rótulos MITRE: 4.2s



# 3. Definição de Caso e Filtragem

> Caso = IP de origem + dia

> Descarta casos com uma técnica só, tal como ele fez ao excluir casos com 1 assinatura ou com 1 evento

> `entrada` é o literal do Sean (só tráfego de entrada); `todas` é sem filtro de direção, só pra comparar

> Alertas sem técnica válida ("T") são descartados

In [151]:
DIRECAO = {"entrada": ["false"], "todas": ["true", "false"]}

In [154]:
with cron("construção do event log e filtragem"):
    LOG, DIAS = {}, {}
    
    for c, df in alertas.items():
        for dir_, vals in DIRECAO.items():
            valido = df["technique"].str.match(PADRAO).fillna(False)
            d = df[df["src_ip"].notna() & df["initiated"].isin(vals) & valido].copy()
            d["dia"] = d["ts"].dt.strftime("%Y-%m-%d")
            d["case"] = d["src_ip"] + "|" + d["dia"]
    
            # para a escolha dos dias (Sean 5.1): calculado antes do filtro, como ele fez
            DIAS[(c, dir_)] = d.groupby("dia").agg(
                alertas=("technique", "size"), ips=("src_ip", "nunique"), tecnicas=("technique", "nunique")
            )
    
            k = d.groupby("case")["technique"].transform("nunique")   # técnicas distintas por caso
            log = d[k > 1].sort_values(["case", "ts"], kind="stable")
            log = log[["case", "ts", "dia", "src_ip", "dst_ip", "technique", "technique_name", "proc_name"]]
            LOG[(c, dir_)] = log
            log.to_parquet(OUT / f"log_{c}_{dir_}.parquet")
    
            print(
                f"{c.upper():>4} {dir_:>7}: {len(d):>9,} alertas -> {len(log):>9,} eventos "
                f"({1 - len(log) / max(len(d), 1):.1%} de redução) | {log['case'].nunique():>2,} casos "
                f"| {log['src_ip'].nunique():>2} IPs | {log['technique'].nunique()} atividades"
            )

 LAB entrada:       587 alertas ->        14 eventos (97.6% de redução) |  3 casos |  2 IPs | 3 atividades
 LAB   todas: 1,175,997 alertas ->   684,904 eventos (41.8% de redução) | 41 casos | 20 IPs | 5 atividades
REAL entrada:       234 alertas ->       134 eventos (42.7% de redução) |  1 casos |  1 IPs | 2 atividades
REAL   todas:     1,676 alertas ->       967 eventos (42.3% de redução) | 24 casos | 14 IPs | 7 atividades

[tempo] construção do event log e filtragem: 9.4s



In [156]:
LOG

{('lab',
  'entrada'):                             case                          ts         dia  \
 18582   10.10.117.130|2022-11-16  2022-11-16 20:48:31.344000  2022-11-16   
 18586   10.10.117.130|2022-11-16  2022-11-16 20:48:33.735000  2022-11-16   
 733122  10.10.117.130|2022-11-18  2022-11-18 11:34:26.564000  2022-11-18   
 733124  10.10.117.130|2022-11-18  2022-11-18 11:34:26.593000  2022-11-18   
 737346  10.10.117.130|2022-11-18  2022-11-18 20:30:25.471000  2022-11-18   
 743265      127.0.0.1|2022-11-18  2022-11-18 06:33:57.441000  2022-11-18   
 743266      127.0.0.1|2022-11-18  2022-11-18 06:33:57.441000  2022-11-18   
 743268      127.0.0.1|2022-11-18  2022-11-18 06:33:57.456000  2022-11-18   
 666509      127.0.0.1|2022-11-18  2022-11-18 11:15:55.541000  2022-11-18   
 666483      127.0.0.1|2022-11-18  2022-11-18 11:17:31.979000  2022-11-18   
 670526      127.0.0.1|2022-11-18  2022-11-18 12:10:43.562000  2022-11-18   
 671132      127.0.0.1|2022-11-18  2022-11-18 12:12:34

In [157]:
DIAS

{('lab',
  'entrada'):             alertas  ips  tecnicas
 dia                               
 2022-11-16      111    8         2
 2022-11-17      209    7         2
 2022-11-18       84    8         3
 2022-11-19       60    7         2
 2022-11-20       19    8         2
 2022-11-21       13    5         2
 2022-11-22       21    3         1
 2022-11-23       10    2         1
 2022-11-24       45    7         1
 2022-11-25       15    3         2,
 ('lab',
  'todas'):             alertas  ips  tecnicas
 dia                               
 2022-11-16    98266   15         3
 2022-11-17   349692   17         4
 2022-11-18   131562   23         5
 2022-11-19    52000   18         4
 2022-11-20   348754   17         3
 2022-11-21   153733   11         4
 2022-11-22     5122    8         3
 2022-11-23     4372    5         4
 2022-11-24    22817   12         4
 2022-11-25     9679   10         3,
 ('real',
  'entrada'):             alertas  ips  tecnicas
 dia                             

## 3.1 Dias selecionados (5.1)
> Dia com mais alertas, mais IPs distintos e mais técnicas distintas

In [155]:
for chave, t in DIAS.items():
    if t.empty:
        print(chave, "vazio")
        continue
    print(chave, "| dias do Sean:", {col: t[col].idxmax() for col in t.columns})
    t

('lab', 'entrada') | dias do Sean: {'alertas': '2022-11-17', 'ips': '2022-11-16', 'tecnicas': '2022-11-18'}


,alertas,ips,tecnicas
dia,,,
2022-11-16,111,8,2
2022-11-17,209,7,2
2022-11-18,84,8,3
2022-11-19,60,7,2
2022-11-20,19,8,2
2022-11-21,13,5,2
2022-11-22,21,3,1
2022-11-23,10,2,1
2022-11-24,45,7,1


('lab', 'todas') | dias do Sean: {'alertas': '2022-11-17', 'ips': '2022-11-18', 'tecnicas': '2022-11-18'}


,alertas,ips,tecnicas
dia,,,
2022-11-16,98266,15,3
2022-11-17,349692,17,4
2022-11-18,131562,23,5
2022-11-19,52000,18,4
2022-11-20,348754,17,3
2022-11-21,153733,11,4
2022-11-22,5122,8,3
2022-11-23,4372,5,4
2022-11-24,22817,12,4


('real', 'entrada') | dias do Sean: {'alertas': '2022-07-05', 'ips': '2022-07-05', 'tecnicas': '2022-07-05'}


,alertas,ips,tecnicas
dia,,,
2022-07-05,134,1,2
2022-07-06,31,1,1
2022-07-07,33,1,1
2022-07-08,36,1,1


('real', 'todas') | dias do Sean: {'alertas': '2022-07-05', 'ips': '2022-07-06', 'tecnicas': '2022-07-05'}


,alertas,ips,tecnicas
dia,,,
2022-06-27,66,7,3
2022-06-28,10,6,1
2022-06-29,14,4,1
2022-07-01,6,5,1
2022-07-03,9,4,1
2022-07-05,578,7,5
2022-07-06,290,14,4
2022-07-07,253,13,3
2022-07-08,330,14,4
